# Resources 
- https://www.youtube.com/watch?v=l8pRSuU81PU&t=14181s (Andrej Karpathy)

In [ ]:
from transformers import GPT2LMHeadModel

In [ ]:
model_hf = GPT2LMHeadModel.from_pretrained("gpt2") # 124M
sd_hf = model_hf.state_dict()

for k, v in sd_hf.items():
    print(k, v.shape)

# Code Explanation

## Key Components

**Embedding Layers:**
- `wte.weight` (50257, 768) - **Token embeddings**: maps each of 50,257 vocabulary tokens to 768-dimensional vectors
- `wpe.weight` (1024, 768) - **Position embeddings**: encodes positions up to 1,024 tokens

**Transformer Blocks (h.0 through h.11):**
This model has **12 identical transformer layers**. Each layer contains:

### 1. First Layer Norm (`ln_1`)
- `weight` and `bias` (768) - normalizes inputs before attention

### 2. Attention Module (`attn`)
- `c_attn.weight` (768, 2304) and `bias` (2304) - computes Query, Key, Value in one shot
  - 2304 = 768 × 3 (for Q, K, V projections)
- `c_proj.weight` (768, 768) and `bias` (768) - output projection after attention

### 3. Second Layer Norm (`ln_2`)
- `weight` and `bias` (768) - normalizes before the feedforward network

### 4. MLP/Feedforward (`mlp`)
- `c_fc.weight` (768, 3072) and `bias` (3072) - expands to 4× hidden size (768 → 3072)
- `c_proj.weight` (3072, 768) and `bias` (768) - projects back down (3072 → 768)

**Output Layer:**
- `ln_f.weight/bias` (768) - final layer normalization
- `lm_head.weight` (50257, 768) - projects hidden states to vocabulary logits

## Model Specs
- **Hidden size:** 768
- **Layers:** 12
- **MLP expansion:** 4× (3072)
- **Max sequence length:** 1024
- **Vocab size:** 50,257

In [ ]:
print(sd_hf["transformer.wpe.weight"].view(-1)[:20])
print(len(sd_hf["transformer.wpe.weight"]))

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

plt.imshow(sd_hf["transformer.wpe.weight"], cmap="gray")

In [ ]:
plt.plot(sd_hf["transformer.wpe.weight"][:, 150])
plt.plot(sd_hf["transformer.wpe.weight"][:, 200])
plt.plot(sd_hf["transformer.wpe.weight"][:, 250])
# this model is kind of "undertrained" - a model which would be better trained would have less noisy embeddings

In [ ]:
plt.imshow(sd_hf["transformer.h.1.attn.c_attn.weight"][:300,:300], cmap="gray")

In [ ]:
from transformers import pipeline, set_seed
generator = pipeline('text-generation', model='gpt2')
set_seed(42)
generator("Hello, I'm a language model,", max_length=30, num_return_sequences=5)

In [ ]:
# let's instead sample manually
import torch
from torch.nn import functional as F

model = GPT2LMHeadModel.from_pretrained("gpt2") # 124M
model.eval()
#model.to('cuda')
torch.manual_seed(42)
torch.cuda.manual_seed(42)
tokens = [15496, 11, 314, 1101, 257, 3303, 2746, 11] # "Hello, I'm a language model,"
tokens = torch.tensor(tokens, dtype=torch.long) # (8,)
tokens = tokens.unsqueeze(0).repeat(5, 1) # (5, 8)
x = tokens.to('cuda')

# generate!
while x.size(1) < 30: # max_length=30
    # forward the model to get the logits
    with torch.no_grad():
        logits = model(x)[0] # (B, T, vocab_size)
        # take the logits at the last position
        logits = logits[:, -1, :] # (B, vocab_size)
        # get the probabilities
        probs = F.softmax(logits, dim=-1)
        # do top-k sampling of 50 (huggingface pipeline default)
        # topk_probs here becomes (5, 50), topk_indices is (5, 50)
        topk_probs, topk_indices = torch.topk(probs, 50, dim=-1)
        # select a token from the top-k probabilities
        # note: multinomial does not demand the input to sum to 1
        ix = torch.multinomial(topk_probs, 1) # (B, 1)
        # gather the corresponding indices
        xcol = torch.gather(topk_indices, -1, ix) # (B, 1)
        # append to the sequence
        x = torch.cat((x, xcol), dim=1)

# print the generated text
import tiktoken
enc = tiktoken.get_encoding('gpt2')
for i in range(5):
    tokens = x[i, :30].tolist()
    decoded = enc.decode(tokens)
    print(">", decoded)

In [11]:
import gensim.downloader
model = gensim.downloader.load('glove-wiki-gigaword-50')

[==================================================] 100.0% 66.0/66.0MB downloaded


In [15]:
tower = model["tower"]  
towers = model["towers"]  
woman = model["woman"]  
man = model["man"]  
uncle = model["uncle"]
aunt = model["aunt"]

man_woman = man-woman
uncle_aunt = uncle-aunt

print(man_woman-uncle_aunt)  # should be close to 0 vector

[ 0.48606402  0.36482993 -0.02332005  0.19448003 -0.50929993 -0.21292996
 -0.16007999 -0.38303     0.32714     0.3425043  -0.48406     0.39434004
 -0.21782005 -0.04819003 -0.27192003  0.14980298  0.28113997 -0.165636
 -0.03161401  0.12234002 -0.08289802  0.47800994  0.501814   -0.27878
  0.2600299   0.41229987 -0.24184996  1.0110301   0.32979    -0.60841995
 -0.11223996  0.36037     0.426646    0.5224371  -0.01406999 -0.28839523
 -0.06432001 -0.125442    0.3485     -0.28442     0.08329701 -0.08606997
  0.29631    -0.24434498 -0.10961005  0.25910303  0.10926002  0.24684995
  0.43481803  0.00340001]
